In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Úvod do primitiv

<Accordion>
<AccordionItem title="Verze balíčků">

Kód na této stránce byl vyvinut pomocí následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## Proč Qiskit zavedl primitiva?
Podobně jako v počátcích klasických počítačů, kdy vývojáři museli přímo manipulovat s registry CPU, rané rozhraní pro QPU jednoduše vracelo surová data z řídicí elektroniky.
To nebyl velký problém, dokud QPU žily v laboratořích a přímý přístup byl umožněn pouze výzkumníkům.
Vzhledem k tomu, že většina vývojářů by neměla a nemusela být obeznámena s převodem těchto surových dat na 0 a 1, Qiskit zavedl `backend.run` jako první abstrakci pro přístup k QPU v cloudu. To umožnilo vývojářům pracovat s důvěrně známým datovým formátem a soustředit se na celkový obraz.

Jak se přístup k QPU stával rozšířenějším a s tím, jak bylo vyvíjeno stále více kvantových algoritmů,
opět se objevila potřeba abstrakce na vyšší úrovni. V reakci na to Qiskit zavedl
rozhraní primitiv, které je optimalizováno pro dva klíčové úkoly při vývoji kvantových algoritmů:
odhad střední hodnoty (`Estimator`) a vzorkování Circuit (`Sampler`). Cílem je opět
pomoci vývojářům více se soustředit na inovaci a méně na převod dat. Rozhraní primitiv nahrazuje rozhraní `backend.run`, protože `Sampler` poskytuje stejný přímý přístup k hardwaru, jaký nabízel `backend.run`.

## Co je to primitivum?
Výpočetní systémy jsou postaveny na více vrstvách abstrakce. Abstrakce ti umožňují soustředit se na
konkrétní úroveň detailu relevantní pro daný úkol. Čím blíže jsi k hardwaru,
tím nižší úroveň abstrakce potřebuješ (například možná budeš muset přesouvat nebo manipulovat s daty na úrovni instrukcí CPU). Čím složitější úkol chceš provést,
tím vyšší úroveň abstrakce bude potřeba (například bys mohl používat programovací knihovnu k provádění
algebraických výpočtů).

V tomto kontextu je *primitivum* nejmenší zpracovatelská instrukce, nejjednodušší stavební kámen, ze kterého
lze vytvořit něco užitečného pro danou úroveň abstrakce.

Nedávný pokrok v kvantových výpočtech zvýšil potřebu pracovat na vyšších úrovních abstrakce.
Jak se oblast posouvá směrem k větším kvantovým zpracovatelským jednotkám (QPU) a složitějším pracovním postupům, přesouvá se zaměření od interakce s jednotlivými signály Qubit k pohledu na kvantová zařízení jako na systémy, které provádějí potřebné úkoly.

Dva nejčastější úkoly pro kvantové počítače jsou vzorkování kvantových stavů a výpočet středních hodnot.
Tyto úkoly motivovaly návrh Qiskit primitiv: **Estimator** a **Sampler**.

- Estimator vypočítává střední hodnoty pozorovatelných veličin vzhledem ke stavům připraveným kvantovými Circuit.
- Sampler vzorkuje výstupní registr z provedení kvantového Circuit.

Stručně řečeno, výpočetní model zavedený Qiskit primitivy posouvá kvantové programování o krok blíže
k tomu, kde se dnes nachází klasické programování, kde je zaměření méně na detaily hardwaru a více na výsledky,
kterých se snažíš dosáhnout.

## Definice a implementace primitiv
Existují dva typy Qiskit primitiv: základní třídy a jejich implementace. Primitiva Estimator a Sampler jsou definována open-source základními třídami primitiv, které žijí v Qiskit SDK (v modulu [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). Poskytovatelé (jako je Qiskit Runtime) mohou tyto základní třídy použít k odvození vlastních implementací Sampler a Estimator. Většina uživatelů bude pracovat s implementacemi poskytovatelů, nikoli se základními primitivy.

### Základní třídy
Primitivy `Base` jsou abstraktní třídy, které definují společné rozhraní pro implementaci primitiv. Všechny ostatní třídy v modulu [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) dědí z těchto základních tříd. Vývojáři by je měli použít, pokud mají zájem o vytvoření vlastního modelu spouštění založeného na primitivech pro konkrétního poskytovatele. Tyto třídy mohou být také užitečné pro ty, kteří chtějí provádět vysoce přizpůsobené zpracování a zjistí, že stávající implementace primitiv jsou pro jejich potřeby příliš jednoduché. Běžní uživatelé základní třídy přímo používat nebudou.

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) a [`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) – Přestože primitiva V1 jsou stále použitelná, tyto průvodce se zaměřují na primitiva V2, protože jsou nejnovější a jsou nejčastěji používaná.

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) a [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) – Referenční primitiva Qiskit se řídí těmito specifikacemi rozhraní.

<span id="implementations"></span>
### Implementace
Všechna primitiva jsou vytvořena ze základních tříd; proto mají stejnou obecnou strukturu a použití. Například formát vstupu pro všechna primitiva Estimator je stejný. Existují však rozdíly v implementacích, které je odlišují.

Toto jsou implementace základních tříd primitiv:

- [Primitiva Qiskit Runtime](/guides/qiskit-runtime-primitives), [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) a [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), poskytují sofistikovanější implementaci (například zahrnují zmírnění chyb) jako cloudovou službu. Tato implementace základních primitiv se používá pro přístup k hardwaru IBM Quantum&reg;.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) a [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) – Referenční implementace primitiv, které používají simulátor zabudovaný do Qiskitu. Jsou postaveny s modulem Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information) a produkují výsledky na základě ideálních simulací stavového vektoru. Jsou dostupné prostřednictvím Qiskitu. Podrobnosti o použití najdeš v části [Přesná simulace s primitivy Qiskit](/guides/simulate-with-qiskit-sdk-primitives).

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) a [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) – Tyto třídy lze použít k "zabalení" libovolného kvantového výpočetního zdroje do primitivu. To ti umožňuje psát kód ve stylu primitiv pro poskytovatele, kteří ještě nemají rozhraní založené na primitivech. Tyto třídy lze používat stejně jako běžný Sampler a Estimator, jen by měly být inicializovány s dodatečným argumentem `backend` pro výběr, na kterém kvantovém počítači spustit výpočet. Jsou dostupné prostřednictvím Qiskitu. Více informací najdeš v průvodci [primitivy pro Backend](/guides/get-started-with-backend-primitives).

## Možnosti
Primitivům lze předávat možnosti pro jejich přizpůsobení podle tvých potřeb. Zatímco rozhraní metody `run()` primitiv je společné pro všechny implementace, jejich možnosti se liší. Prostuduj si reference API pro konkrétní implementaci primitivu, abys zjistil/a, jaké možnosti podporuje.

Například viz témata [Možnosti Estimatoru](/guides/estimator-options) a [Možnosti Sampleru](/guides/sampler-options) pro možnosti primitiv Qiskit Runtime, nebo viz [reference API Qiskit Aer](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) pro možnosti primitiv Qiskit Aer.

## Výhody Qiskit primitiv
S primitivy mohou uživatelé Qiskitu psát kvantový kód pro konkrétní QPU, aniž by museli explicitně
spravovat každý detail. Také díky dodatečné vrstvě abstrakce by ti mohlo být snadnější
přistupovat k pokročilým hardwarovým schopnostem daného poskytovatele. Například s primitivy Qiskit Runtime
můžeš využít nejnovější pokroky v zmírňování a potlačování chyb přepínáním možností jako je [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) primitivu, namísto budování vlastní implementace těchto technik.

Pro poskytovatele hardwaru znamená nativní implementace primitiv, že svým uživatelům můžeš poskytnout přístup k funkcím hardwaru "hned po vybalení", jako jsou pokročilé techniky post-processingu. Pro tvé uživatele je tak snazší využívat nejlepší schopnosti tvého hardwaru.

## Další kroky
> **Tip:** - Porozuměj [vstupům a výstupům primitiv](/guides/primitive-input-output).
> - Prostuduj si podrobné [příklady](/guides/simulate-with-qiskit-sdk-primitives).
> - Procvič si primitiva prací s [lekcí o funkci nákladů](/learning/courses/variational-algorithm-design/cost-functions) v IBM Quantum Learning.
> - Prostuduj si [Vytvoření poskytovatele](/guides/create-a-provider), kde se dozvíš, jak implementovat vlastní primitiva Sampler a Estimator.
> - Viz [reference API](https://docs.quantum.ibm.com/api/qiskit/primitives).
> - Přečti si [Migrace na primitiva V2](/guides/v2-primitives).
> - Dozvěz se více o [primitivech Qiskit Runtime](/guides/qiskit-runtime-primitives), které se používají pro spouštění Circuit na IBM QPU.